In [48]:
import requests_cache
import json
import pandas as pd
from time import sleep

In [49]:
session = requests_cache.CachedSession(
    cache_name='Extra', use_cache_dir=True, expire_after=604800)

In [50]:
def all_matches_data(url):
    response = session.get(url)
    data = json.loads(response.content)
    ids = [box['id'] for box in data['matches']
           ['allMatches'] if box['status']['started'] == True]
    return ids

In [51]:
def match_details(ids):
    all_teams = []
    ids_count = len(ids)
    for id in ids:
        base_data = {}
        home_data = {}
        away_data = {}
        url = f'https://www.fotmob.com/api/matchDetails?matchId={id}'
        print(url)
        response = session.get(url)
        data = json.loads(response.content)
        
        counter = 0
        if 'content' in data:
            check_for_none = data['content']['stats']

            if check_for_none is not None and check_for_none['Periods'] is not None and check_for_none['Periods']['All'] is not None:
                if len(data['content']['stats']['Periods']['All']['stats']) > 2:
                    base_data['match_date'] = data['general']['matchTimeUTCDate'].split('T')[0]
                    base_data['league_name'] = data['general']['leagueName']
                    base_data['matchid'] = data['general']['matchId']
                    base_data['home_team'] = data['general']['homeTeam']['name']
                    base_data['away_team'] = data['general']['awayTeam']['name']
                    # Check if 'voteResult' exists in the dictionary
                    if 'voteResult' in data['content']['matchFacts']['poll']:
                        votes = [item['Votes'] for item in data['content']['matchFacts']['poll']['voteResult']['Votes'] if len(item['Votes']) == 3]
                        # print(votes)
                    else:
                        votes = 0
                    base_data['home_probability_win'] = votes[0][0] / \
                        sum(votes[0]) if isinstance(votes[0], list) else None
                    base_data['draw_probability'] = votes[0][1] / \
                        sum(votes[0]) if isinstance(votes[0], list) else None
                    base_data['away_probability_win'] = votes[0][2] / \
                        sum(votes[0]) if isinstance(votes[0], list) else None

                    base_data['home_goals'] = int(data['header']['teams'][0]['score'])
                    base_data['away_goals'] = int(data['header']['teams'][1]['score'])
                    base_data['ftr'] = 0 if base_data['home_goals'] > base_data['away_goals'] else 1 if base_data['home_goals'] == base_data['away_goals'] else 2
                    for information in data['content']['stats']['Periods']['All']['stats']:
                        for details in information['stats']:
                            if details['title'] == 'Ball possession': # the percentage symbol should be ommited
                                home_data['home_ball_possession'] = details['stats'][0]
                                away_data['away_ball_possession'] = details['stats'][1]
                            elif details['title'] == "Big chances":
                                home_data['home_big_chances'] = details['stats'][0]
                                away_data['away_big_chances'] = details['stats'][1]
                            elif details['title'] == "Big chances missed":
                                home_data['home_big_misses'] = details['stats'][0]
                                away_data['away_big_misses'] = details['stats'][1]
                            elif details['title'] == "Fouls committed":
                                home_data['home_fouls_commited'] = details['stats'][0]
                                away_data['away_fouls_commited'] = details['stats'][1]
                            elif details['title'] == "Corners":
                                home_data['home_corners'] = details['stats'][0]
                                away_data['away_corners'] = details['stats'][1]
                            elif details['key'] == "total_shots":
                                home_data['home_total_shots'] = details['stats'][0]
                                away_data['away_total_shots'] = details['stats'][1]
                            elif details['title'] == "Shots off target":
                                home_data['home_shots_off_target'] = details['stats'][0]
                                away_data['away_shots_off_target'] = details['stats'][1]
                            elif details['title'] == "Shots on target":
                                home_data['home_shots_on_target'] = details['stats'][0]
                                away_data['away_shots_on_target'] = details['stats'][1]
                            elif details['title'] == "Blocked shots":
                                home_data['home_blocked_shots'] = details['stats'][0]
                                away_data['away_blocked_shots'] = details['stats'][1]
                            elif details['title'] == "Hit woodwork":
                                home_data['home_hit_woodwork'] = details['stats'][0]
                                away_data['away_hit_woodwork'] = details['stats'][1]
                            elif details['title'] == "Shots inside box":
                                home_data['home_shots_inside_box'] = details['stats'][0]
                                away_data['away_shots_inside_box'] = details['stats'][1]
                            elif details['title'] == "Shots outside box":
                                home_data['home_shots_outside_box'] = details['stats'][0]
                                away_data['away_shots_outside_box'] = details['stats'][1]
                            elif details['title'] == "Passes" and details['type'] == 'text':
                                home_data['home_passes'] = details['stats'][0]
                                away_data['away_passes'] = details['stats'][1]
                            elif details['title'] == "Accurate passes":
                                values = [int(item.split()[1][1:-2]) if isinstance(item, str) else item for item in details['stats']]
                                home_data['home_accurate_passes'] = values[0]
                                away_data['away_accurate_passes'] = values[1]
                            elif details['title'] == "Own half":
                                home_data['home_own_half'] = details['stats'][0]
                                away_data['away_own_half'] = details['stats'][1]
                            elif details['title'] == "Opposition half":
                                home_data['home_opposition_half'] = details['stats'][0]
                                away_data['away_opposition_half'] = details['stats'][1]
                            elif details['title'] == "Accurate long balls":
                                values = [int(item.split()[1][1:-2]) if isinstance(item, str) else item for item in details['stats']]
                                home_data['home_accurate_long_balls'] = values[0]
                                away_data['away_accurate_long_balls'] = values[1]
                            elif details['title'] == "Accurate crosses":
                                values = [int(item.split()[1][1:-2]) if isinstance(item, str) else item for item in details['stats']]
                                home_data['home_accurate_crosses'] = values[0]
                                away_data['away_axxurate_crosses'] = values[1]                                    
                            elif details['title'] == "Throws":
                                home_data['home_throws'] = details['stats'][0]
                                away_data['away_throws'] = details['stats'][1]
                            elif details['title'] == "Offsides":
                                home_data['home_offsides'] = details['stats'][0]
                                away_data['away_offsides'] = details['stats'][1]
                            elif details['title'] == "Tackles won":
                                values = [int(item.split()[1][1:-2]) if isinstance(item, str) else item for item in details['stats']]
                                home_data['home_tackles_won'] = values[0]
                                away_data['away_tackles_won'] = values[1]                                    
                            elif details['title'] == "Interceptions":
                                home_data['home_interceptions'] = details['stats'][0]
                                away_data['away_interceptions'] = details['stats'][1]
                            elif details['title'] == "Blocks":
                                home_data['home_blocks'] = details['stats'][0]
                                away_data['away_blocks'] = details['stats'][1]
                            elif details['title'] == "Clearances":
                                home_data['home_clearances'] = details['stats'][0]
                                away_data['away_clearances'] = details['stats'][1]
                            elif details['title'] == "Keeper saves":
                                home_data['home_keeper_save'] = details['stats'][0]
                                away_data['away_keeper_save'] = details['stats'][1]
                            elif details['title'] == "Duels won":
                                home_data['home_duels_won'] = details['stats'][0]
                                away_data['away_duels_won'] = details['stats'][1]
                            elif details['title'] == "Ground duels won":
                                values = [int(item.split()[1][1:-2]) if isinstance(item, str) else item for item in details['stats']]
                                home_data['home_geround_duels_won'] = values[0]
                                away_data['away_geround_duels_won'] = values[1] 
                            elif details['title'] == "Aerial duels won":
                                values = [int(item.split()[1][1:-2]) if isinstance(item, str) else item for item in details['stats']]
                                home_data['home_aerial_duels_won'] = values[0]
                                away_data['away_aerial_duels_won'] = values[1]
                            elif details['title'] == "Successful dribbles":
                                values = [int(item.split()[1][1:-2]) if isinstance(item, str) else item for item in details['stats']]
                                home_data['home_successful_dribbles'] = values[0]
                                away_data['away_successful_dribbles'] = values[1] 
                            elif details['title'] == "Yellow cards":
                                home_data['home_yellow_cards'] = details['stats'][0]
                                away_data['away_yellow_cards'] = details['stats'][1]
                            elif details['title'] == "Red cards":
                                home_data['home_red_cards'] = details['stats'][0]
                                away_data['away_red_cards'] = details['stats'][1]
                    each_team = {**base_data, **home_data, **away_data}
                    all_teams.append(each_team)
                else:
                    print(f"The id {id} doesn't have enough information")
                    counter += 1
            else:
                print(f'The stats for id: {id} is None. maybe the match is not started yet.')
                counter += 1
        else:
            print('There is no data in the content.')
            counter += 1
        if int((len(ids) / 100)) * 40 == counter:
            continue
        sleep(10)
        if ids_count % 15 == 0:
            sleep(60)
        ids_count -= 1
        print('The remain ids are ',ids_count)
        print('-' * 60)
    return all_teams                   

In [52]:
seasons = [
    '2022%2F2023',
    '2021%2F2022',
    '2020%2F2021',
    '2019%2F2020',
    '2018%2F2019',
    '2017%2F2018',
    '2016%2F2017',
    '2015%2F2016',
    '2023',
    '2022',
    '2021',
    '2020',
    '2019',
    '2018',
    '2017',
    '2016'
    ]

leagues = {}

leagues['argentinalp'] = ['2023', '2022', '2021', '2019%2F2020', '2018%2F2019']  # 2024

leagues['copaargentina'] = ['2020', '2022', '2023'] # 2024
leagues['copa_de_laliga_argentina'] = ['2021%2F2022', '2022', '2023'] # 2024

leagues['A_league_australia'] = ['2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023'] # 2023%2F2024

leagues['A_league_australia_women'] = ['2021%2F2022', '2022%2F2023'] # 2023%2F2024

leagues['bundesliga_austria'] = ['2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['first_division_a'] = ['2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['serieA_brazil'] = ['2017', '2018', '2019', '2020', '2021', '2022', '2023'] # 2024

leagues['serieB_brazil'] = ['2020', '2021', '2022', '2023'] # 2024

leagues['paulista_A1'] = ['2023'] # 2024

leagues['primier_league_canada'] = ['2019', '2020', '2021', '2022', '2023']  # 2024

leagues['canadian_championship'] = ['2019', '2020', '2021', '2022', '2023']  # 2024

leagues['primera_division_chile'] = ['2021', '2022', '2023']  # 2024

leagues['cup_chile'] = ['2021', '2022', '2023']  # 2024

leagues['super_league_china'] = ['2019', '2020', '2021', '2022', '2023']  # 2024   

leagues['primera_A_colombia'] = ['2021+%2F+Apertura', '2021+%2F+Clausura', '2022+%2F+Apertura', '2022+%2F+Clausura', '2023+%2F+Apertura', '2023+%2F+Clausura']

leagues['HNL_croatia'] = ['2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['superligaen_denmark'] = ['2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['onedivision_denmark'] = ['2022%2F2023']  # 2023%2F2024

leagues['primier_league_egypt'] = ['2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['primier_league_england'] = ['2016%2F2017', '2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['championship_england'] = ['2015%2F2016','2016%2F2017', '2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['league_one_england'] = ['2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['league_two_england'] = ['2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['FA_cup_england'] = ['2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['efl_cup_england'] = ['2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['wsl_england'] = ['2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['league1_france'] = ['2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['league2_france'] = ['2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['copa_de_france'] = ['2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['bundesliga_germany'] = ['2016%2F2017', '2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['bundesliga2_germany'] = ['2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['liga3_germany'] = ['2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['dfb_pokal_germany'] = ['2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['Frauen_Bundesliga'] = ['2022%2F2023']  # 2023%2F2024

leagues['super_league1_greece'] = ['2022%2F2023']  # 2023%2F2024

leagues['besta_deildin_iceland'] = ['2023'] #2024

leagues['super_league_india'] = ['2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['premier_division_ireland'] = ['2022', '2023']  # 2024

leagues['serieA_italy'] = ['2016%2F2017', '2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['serieB_italy'] = ['2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['coppa_italia_italy'] = ['2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['jleague_japan'] = ['2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['ligamx_mexico'] = ['2018%2F2019+%2F+Apertura', '2018%2F2019+%2F+Clausura',  '2019%2F2020+%2F+Apertura', '2019%2F2020+%2F+Clausura', '2020%2F2021+%2F+Apertura', '2020%2F2021+%2F+Clausura', '2021%2F2022+%2F+Apertura','2021%2F2022+%2F+Clausura', '2022%2F2023+%2F+Apertura', '2022%2F2023+%2F+Clausura', '2023%2F2024+%2F+Apertura', '2023%2F2024+%2F+Clausura']

leagues['Eredivisie_netherland'] = ['2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021','2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['eerste_divisie_netherland'] = ['2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['knvb_cup_netherland'] = ['2023%2F2024']

leagues['elisteserien'] = ['2017', '2018', '2019', '2020', '2021', '2022', '2023'] # 2024

leagues['Ekstraklasa_poland'] = ['2022%2F2023']  # 2023%2F2024

leagues['liga_portugal'] = ['2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['premier_league_russia'] = ['2018%2F2019', '2019%2F2020', '2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['kings_cup_KSA'] = ['2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['premiership_scotland'] = ['2020%2F2021', '2021%2F2022', '2022%2F2023']  # 2023%2F2024

leagues['k_league1_skorea'] = ['2022', '2023']  # 2024

leagues['k_league2_skorea'] = ['2022', '2023']  # 2024

leagues['laliga_spain'] = ['2016%2F2017', '2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021',
                '2021%2F2022', '2022%2F2023']  # 2023%2F2024
leagues['laliga2_spain'] = ['2018%2F2019', '2019%2F2020', '2020%2F2021',
                '2021%2F2022', '2022%2F2023']  # 2023%2F2024
leagues['laligaf_spain'] = ['2022%2F2023']  # 2023%2F2024
leagues['copa_del_rey_spain'] = ['2020%2F2021',
                      '2021%2F2022', '2022%2F2023']  # 2023%2F2024
leagues['Allsvenskan_sweeden'] = ['2018', '2019', '2020', '2021', '2022', '2023']  # 2024
leagues['super_league_switzerland'] = ['2018%2F2019', '2019%2F2020', '2020%2F2021',
                '2021%2F2022', '2022%2F2023']  # 2023%2F2024
leagues['thai_league_thailand'] = ['2022%2F2023']  # 2023%2F2024
leagues['super_league_turkey'] = ['2018%2F2019', '2019%2F2020', '2020%2F2021',
                       '2021%2F2022', '2022%2F2023']  # 2023%2F2024
leagues['proleagueUAE'] = ['2020%2F2021',
                '2021%2F2022', '2022%2F2023']  # 2023%2F2024
leagues['MLS_USA'] = ['2016','2017','2018', '2019', '2020', '2021', '2022', '2023']  # 2024
leagues['usl_championship_usa'] = ['2019', '2020', '2021', '2022', '2023']  # 2024
leagues['usl_league_one'] = ['2019', '2020', '2021', '2022', '2023']  # 2024
leagues['open_cup_usa'] = ['2019', '2020', '2021', '2022', '2023']  # 2024
leagues['nwsl_usa'] = ['2019', '2020', '2021', '2022', '2023']  # 2024
leagues['mls_nextpro_usa'] = ['2022', '2023']  # 2024
leagues['champions_league'] = ['2016%2F2017', '2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021',
                               '2021%2F2022', '2022%2F2023']  # 2023%2F2024
leagues['europa_league'] = ['2016%2F2017', '2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021',
                            '2021%2F2022', '2022%2F2023']  # 2023%2F2024
leagues['afc_champions_league'] = ['2020', '2021', '2022']  # 2023-2024 !!!!!!
leagues['asian_cup'] = ['2023']
leagues['caf_champions_league'] = ['2020%2F2021',
                                   '2021%2F2022', '2022%2F2023']  # 2023%2F2024
leagues['caf_confed_cup'] = ['2020%2F2021',
                             '2021%2F2022', '2022%2F2023']  # 2023%2F2024
leagues['cocacaf_champ'] = ['2019', '2020', '2021', '2022']  # 2024
leagues['concacaf_gold'] = ['2021', '2023']
leagues['conmebol'] = ['2022']
leagues['copa_america'] = ['2019', '2021']  # 2024
leagues['copa_libertadores'] = ['2019', '2020', '2021', '2022']  # 2024
leagues['copa_sudamericana'] = ['2020', '2021', '2022']  # 2024
leagues['euro'] = ['2016', '2018-2019', '2020', '2022-2023']  # 2024
leagues['eurou21'] = ['2023']
leagues['europa_conference'] = ['2021%2F2022', '2022%2F2023']  # 2023%2F2024
leagues['fifa_clup_world_cup'] = ['2021', '2022', '2023']
leagues['leagues_cup_int'] = ['2023']
leagues['uefa_nation_leagueA'] = ['2020%2F2021', '2022%2F2023']  # 2024%2F2025
leagues['uefa_nation_leagueB'] = ['2020%2F2021', '2022%2F2023']  # 2024%2F2025
leagues['uefa_nation_leagueC'] = ['2020%2F2021', '2022%2F2023']  # 2024%2F2025
leagues['uefa_nation_leagueD'] = ['2020%2F2021', '2022%2F2023']  # 2024%2F2025
leagues['uefa_womens_euro'] = ['2022']
leagues['womens_champions_league'] = ['2022%2F2023']  # 2023%2F2024
leagues['womens_world_cuup'] = ['2019', '2023']
leagues['fiffa_world_cup'] = ['2018', '2022']
leagues['world_cup_qualification_concacaf'] = [
    '2015%2F2017', '2021%2F2022']  # 2024%2F2025
leagues['world_cup_qualification_conmebol'] = [
    '2015%2F2017', '2020%2F2022', '2023%2F2025']
leagues['world_cup_qualification_uefa'] = ['2016%2F2017', '2021%2F2022']

In [44]:
# international = {
#     'champions_league': f'https://www.fotmob.com/api/leagues?id=42&ccode3=UK&season={season}',
#     'europa_league': f'https://www.fotmob.com/api/leagues?id=73&ccode3=UK&season={season}',
#     'afc_champions_league': f'https://www.fotmob.com/api/leagues?id=525&ccode3=UK&season={season}',
#     'asian_cup': f'https://www.fotmob.com/api/leagues?id=290&ccode3=UK&season={season}',
#     'caf_champions_league': f'https://www.fotmob.com/api/leagues?id=526&ccode3=UK&season={season}',
#     'caf_confed_cup': f'https://www.fotmob.com/api/leagues?id=9468&ccode3=UK&season={season}',
#     'cocacaf_champ': f'https://www.fotmob.com/api/leagues?id=297&ccode3=UK&season={season}',
#     'concacaf_gold': f'https://www.fotmob.com/api/leagues?id=298&ccode3=UK&season={season}',
#     'conmebol': f'https://www.fotmob.com/api/leagues?id=10304&ccode3=UK&season={season}',
#     'copa_america': f'https://www.fotmob.com/api/leagues?id=44&ccode3=UK&season={season}',
#     'copa_libertadores': f'https://www.fotmob.com/api/leagues?id=45&ccode3=UK&season={season}',
#     'copa_sudamericana' : f'https://www.fotmob.com/api/leagues?id=299&ccode3=UK&season={season}',
#     'euro': f'https://www.fotmob.com/api/leagues?id=50&ccode3=UK&season={season}',
#     'eurou21': f'https://www.fotmob.com/api/leagues?id=288&ccode3=UK&season={season}',
#     'europa_conference': f'https://www.fotmob.com/api/leagues?id=10216&ccode3=UK&season={season}',
#     'fifa_clup_world_cup': f'https://www.fotmob.com/api/leagues?id=78&ccode3=UK&season={season}',
#     'api/leagues?id=cup_int': f'https://www.fotmob.com/api/leagues?id=10043&ccode3=UK&season={season}',
#     'uefa_nation_leagueA': f'https://www.fotmob.com/api/leagues?id=9806&ccode3=UK&season={season}',
#     'uefa_nation_leagueB': f'https://www.fotmob.com/api/leagues?id=9807&ccode3=UK&season={season}',
#     'uefa_nation_leagueC': f'https://www.fotmob.com/api/leagues?id=9808&ccode3=UK&season={season}',
#     'uefa_nation_leagueD': f'https://www.fotmob.com/api/leagues?id=9809&ccode3=UK&season={season}',
#     'uefa_womens_euro': f'https://www.fotmob.com/api/leagues?id=292&ccode3=UK&season={season}',
#     'womens_champions_league': f'https://www.fotmob.com/api/leagues?id=9375&ccode3=UK&season={season}',
#     'womens_world_cuup': f'https://www.fotmob.com/api/leagues?id=76&ccode3=UK&season={season}',
#     'fiffa_world_cup': f'https://www.fotmob.com/api/leagues?id=77&ccode3=UK&season={season}',
#     'world_cup_qualification_concacaf': f'https://www.fotmob.com/api/leagues?id=10198&ccode3=UK&season={season}',
#     'world_cup_qualification_conmebol': f'https://www.fotmob.com/api/leagues?id=10199&ccode3=UK&season={season}',
#     'world_cup_qualification_uefa': f'https://www.fotmob.com/api/leagues?id=10195&ccode3=UK&season={season}',

# }

# leagues['champions_league'] = ['2016%2F2017', '2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021',
#                     '2021%2F2022', '2022%2F2023']  # 2023%2F2024
# leagues['europa_league'] = ['2016%2F2017', '2017%2F2018', '2018%2F2019', '2019%2F2020', '2020%2F2021',
#                     '2021%2F2022', '2022%2F2023']  # 2023%2F2024
# leagues['afc_champions_league'] = ['2020', '2021', '2022'] # 2023-2024 !!!!!!
# leagues['asian_cup'] = ['2023']
# leagues['caf_champions_league'] = ['2020%2F2021',
#                     '2021%2F2022', '2022%2F2023']  # 2023%2F2024
# leagues['caf_confed_cup'] = ['2020%2F2021',
#                   '2021%2F2022', '2022%2F2023']  # 2023%2F2024
# leagues['cocacaf_champ'] = ['2019','2020', '2021', '2022'] #2024
# leagues['concacaf_gold'] = ['2021', '2023']
# leagues['conmebol'] = ['2022']
# leagues['copa_america'] = ['2019', '2021'] #2024
# leagues['copa_libertadores'] = ['2019', '2020', '2021', '2022'] #2024
# leagues['copa_sudamericana'] = ['2020', '2021', '2022']  # 2024
# leagues['euro'] = ['2016', '2018-2019', '2020', '2022-2023'] #2024
# leagues['eurou21'] = ['2023']
# leagues['europa_conference'] = ['2021%2F2022', '2022%2F2023']  # 2023%2F2024
# leagues['fifa_clup_world_cup'] = ['2021', '2022', '2023']
# leagues['leagues_cup_int'] = ['2023']
# leagues['uefa_nation_leagueA'] = ['2020%2F2021', '2022%2F2023'] # 2024%2F2025
# leagues['uefa_nation_leagueB'] = ['2020%2F2021', '2022%2F2023'] # 2024%2F2025
# leagues['uefa_nation_leagueC'] = ['2020%2F2021', '2022%2F2023'] # 2024%2F2025
# leagues['uefa_nation_leagueD'] = ['2020%2F2021', '2022%2F2023'] # 2024%2F2025
# leagues['uefa_womens_euro'] = ['2022']
# leagues['womens_champions_league'] = ['2022%2F2023'] # 2023%2F2024
# leagues['womens_world_cuup'] = ['2019', '2023']
# leagues['fiffa_world_cup'] = ['2018', '2022']
# leagues['world_cup_qualification_concacaf'] = ['2015%2F2017', '2021%2F2022'] # 2024%2F2025
# leagues['world_cup_qualification_conmebol'] = ['2015%2F2017', '2020%2F2022', '2023%2F2025']
# leagues['world_cup_qualification_uefa'] = ['2016%2F2017', '2021%2F2022']

In [45]:
# leagues_per_season = {}
# for season in seasons:
#     hml = []
#     leagues_pt = {
#         'argentinalp': f'https://www.fotmob.com/api/leagues?id=112&ccode3=UK&season={season}',
#         'copaargentina': f'https://www.fotmob.com/api/leagues?id=9305&ccode3=UK&season={season}',
#         'copa_de_laliga_argentina': f'https://www.fotmob.com/api/leagues?id=10007&ccode3=UK&season={season}',
#         'A_league_australia': f'https://www.fotmob.com/api/leagues?id=113&ccode3=UK&season={season}',
#         'A_league_australia_women': f'https://www.fotmob.com/api/leagues?id=9495&ccode3=UK&season={season}',
#         'bundesliga_austria': f'https://www.fotmob.com/api/leagues?id=38&ccode3=UK&season={season}',
#         'first_division_a': f'https://www.fotmob.com/api/leagues?id=40&ccode3=UK&season={season}',
#         'serieA_brazil': f'https://www.fotmob.com/api/leagues?id=268&ccode3=UK&season={season}',
#         'serieB_brazil': f'https://www.fotmob.com/api/leagues?id=8814&ccode3=UK&season={season}',
#         'paulista_A1': f'https://www.fotmob.com/api/leagues?id=10244&ccode3=UK&season={season}',
#         'primier_league_canada': f'https://www.fotmob.com/api/leagues?id=9986&ccode3=UK&season={season}',
#         'canadian_championship': f'https://www.fotmob.com/api/leagues?id=9837&ccode3=UK&season={season}',
#         'primera_division_chile': f'https://www.fotmob.com/api/leagues?id=273&ccode3=UK&season={season}',
#         'cup_chile': f'https://www.fotmob.com/api/leagues?id=9091&ccode3=UK&season={season}',
#         'super_league_china': f'https://www.fotmob.com/api/leagues?id=120&ccode3=UK&season={season}',
#         'primera_A_colombia': f'https://www.fotmob.com/api/leagues?id=274&ccode3=UK&season={season}',
#         'HNL_croatia': f'https://www.fotmob.com/api/leagues?id=252&ccode3=UK&season={season}',
#         'superligaen_denmark': f'https://www.fotmob.com/api/leagues?id=46&ccode3=UK&season={season}',
#         'onedivision_denmark': f'https://www.fotmob.com/api/leagues?id=85&ccode3=UK&season={season}',
#         'primier_league_egypt': f'https://www.fotmob.com/api/leagues?id=519&ccode3=UK&season={season}',
#         'primier_league_england': f'https://www.fotmob.com/api/leagues?id=47&ccode3=UK&season={season}',
#         'championship_england': f'https://www.fotmob.com/api/leagues?id=48&ccode3=UK&season={season}',
#         'league_one_england': f'https://www.fotmob.com/api/leagues?id=108&ccode3=UK&season={season}',
#         'league_two_england': f'https://www.fotmob.com/api/leagues?id=109&ccode3=UK&season={season}',
#         'FA_cup_england': f'https://www.fotmob.com/api/leagues?id=132&ccode3=UK&season={season}',
#         'efl_cup_england': f'https://www.fotmob.com/api/leagues?id=133&ccode3=UK&season={season}',
#         'wsl_england': f'https://www.fotmob.com/api/leagues?id=9227&ccode3=UK&season={season}',
#         'league1_france': f'https://www.fotmob.com/api/leagues?id=53&ccode3=UK&season={season}',
#         'league1_france': f'https://www.fotmob.com/api/leagues?id=110&ccode3=UK&season={season}',
#         'copa_de_france': f'https://www.fotmob.com/api/leagues?id=134&ccode3=UK&season={season}',
#         'bundesliga_germany': f'https://www.fotmob.com/api/leagues?id=54&ccode3=UK&season={season}',
#         'bundesliga2_germany': f'https://www.fotmob.com/api/leagues?id=146&ccode3=UK&season={season}',
#         'liga3_germany': f'https://www.fotmob.com/api/leagues?id=208&ccode3=UK&season={season}',
#         'dfb_pokal_germany': f'https://www.fotmob.com/api/leagues?id=209&ccode3=UK&season={season}',
#         'Frauen_Bundesliga': f'https://www.fotmob.com/api/leagues?id=9676&ccode3=UK&season={season}',
#         'super_league1_greece': f'https://www.fotmob.com/api/leagues?id=135&ccode3=UK&season={season}',
#         'besta_deildin_iceland': f'https://www.fotmob.com/api/leagues?id=215&ccode3=UK&season={season}',
#         'super_league_india': f'https://www.fotmob.com/api/leagues?id=9478&ccode3=UK&season={season}',
#         'premier_division_ireland':  f'https://www.fotmob.com/api/leagues?id=126&ccode3=UK&season={season}',
#         'serieA_italy': f'https://www.fotmob.com/api/leagues?id=55&ccode3=UK&season={season}',
#         'serieB_italy': f'https://www.fotmob.com/api/leagues?id=86&ccode3=UK&season={season}',
#         'coppa_italia_italy': f'https://www.fotmob.com/api/leagues?id=141&ccode3=UK&season={season}',
#         'jleague_japan': f'https://www.fotmob.com/api/leagues?id=223&ccode3=UK&season={season}',
#         'ligamx_mexico': f'https://www.fotmob.com/api/leagues?id=230&ccode3=UK&season={season}',
#         'Eredivisie_netherland': f'https://www.fotmob.com/api/leagues?id=57&ccode3=UK&season={season}',
#         'eerste_divisie_netherland': f'https://www.fotmob.com/api/leagues?id=111&ccode3=UK&season={season}',
#         'knvb_cup_netherland': f'https://www.fotmob.com/api/leagues?id=235&ccode3=UK&season={season}',
#         'elisteserien': f'https://www.fotmob.com/api/leagues?id=59&ccode3=UK&season={season}',
#         'Ekstraklasa_poland': f'https://www.fotmob.com/api/leagues?id=196&ccode3=UK&season={season}',
#         'liga_portugal': f'https://www.fotmob.com/api/leagues?id=61&ccode3=UK&season={season}',
#         'premier_league_russia': f'https://www.fotmob.com/api/leagues?id=63&ccode3=UK&season={season}',
#         'kings_cup_KSA': f'https://www.fotmob.com/api/leagues?id=9942&ccode3=UK&season={season}',
#         'premiership_scotland': f'https://www.fotmob.com/api/leagues?id=64&ccode3=UK&season={season}',
#         'k_league1_skorea': f'https://www.fotmob.com/api/leagues?id=9080&ccode3=UK&season={season}',
#         'k_league2_skorea': f'https://www.fotmob.com/api/leagues?id=9116&ccode3=UK&season={season}',
#         'laliga_spain': f'https://www.fotmob.com/api/leagues?id=87&ccode3=UK&season={season}',
#         'laliga2_spain': f'https://www.fotmob.com/api/leagues?id=140&ccode3=UK&season={season}',
#         'laligaf_spain': f'https://www.fotmob.com/api/leagues?id=9907&ccode3=UK&season={season}',
#         'copa_del_rey_spain': f'https://www.fotmob.com/api/leagues?id=138&ccode3=UK&season={season}',
#         'Allsvenskan_sweeden': f'https://www.fotmob.com/api/leagues?id=67&ccode3=UK&season={season}',
#         'super_league_switzerland': f'https://www.fotmob.com/api/leagues?id=69&ccode3=UK&season={season}',
#         'thai_league_thailand': f'https://www.fotmob.com/api/leagues?id=8984&ccode3=UK&season={season}',
#         'super_league_turkey': f'https://www.fotmob.com/api/leagues?id=71&ccode3=UK&season={season}',
#         'proleagueUAE': f'https://www.fotmob.com/api/leagues?id=538&ccode3=UK&season={season}',
#         'MLS_USA': f'https://www.fotmob.com/api/leagues?id=130&ccode3=UK&season={season}',
#         'usl_championship_usa': f'https://www.fotmob.com/api/leagues?id=8972&ccode3=UK&season={season}',
#         'usl_league_one': f'https://www.fotmob.com/api/leagues?id=9296&ccode3=UK&season={season}',
#         'open_cup_usa': f'https://www.fotmob.com/api/leagues?id=9441&ccode3=UK&season={season}',
#         'nwsl_usa': f'https://www.fotmob.com/api/leagues?id=9134&ccode3=UK&season={season}',
#         'mls_nextpro_usa': f'https://www.fotmob.com/api/leagues?id=10282&ccode3=UK&season={season}',
#         'champions_league': f'https://www.fotmob.com/api/leagues?id=42&ccode3=UK&season={season}',
#         'europa_league': f'https://www.fotmob.com/api/leagues?id=73&ccode3=UK&season={season}',
#         'afc_champions_league': f'https://www.fotmob.com/api/leagues?id=525&ccode3=UK&season={season}',
#         'asian_cup': f'https://www.fotmob.com/api/leagues?id=290&ccode3=UK&season={season}',
#         'caf_champions_league': f'https://www.fotmob.com/api/leagues?id=526&ccode3=UK&season={season}',
#         'caf_confed_cup': f'https://www.fotmob.com/api/leagues?id=9468&ccode3=UK&season={season}',
#         'cocacaf_champ': f'https://www.fotmob.com/api/leagues?id=297&ccode3=UK&season={season}',
#         'concacaf_gold': f'https://www.fotmob.com/api/leagues?id=298&ccode3=UK&season={season}',
#         'conmebol': f'https://www.fotmob.com/api/leagues?id=10304&ccode3=UK&season={season}',
#         'copa_america': f'https://www.fotmob.com/api/leagues?id=44&ccode3=UK&season={season}',
#         'copa_libertadores': f'https://www.fotmob.com/api/leagues?id=45&ccode3=UK&season={season}',
#         'copa_sudamericana': f'https://www.fotmob.com/api/leagues?id=299&ccode3=UK&season={season}',
#         'euro': f'https://www.fotmob.com/api/leagues?id=50&ccode3=UK&season={season}',
#         'eurou21': f'https://www.fotmob.com/api/leagues?id=288&ccode3=UK&season={season}',
#         'europa_conference': f'https://www.fotmob.com/api/leagues?id=10216&ccode3=UK&season={season}',
#         'fifa_clup_world_cup': f'https://www.fotmob.com/api/leagues?id=78&ccode3=UK&season={season}',
#         'leagues_cup_int': f'https://www.fotmob.com/api/leagues?id=10043&ccode3=UK&season={season}',
#         'uefa_nation_leagueA': f'https://www.fotmob.com/api/leagues?id=9806&ccode3=UK&season={season}',
#         'uefa_nation_leagueB': f'https://www.fotmob.com/api/leagues?id=9807&ccode3=UK&season={season}',
#         'uefa_nation_leagueC': f'https://www.fotmob.com/api/leagues?id=9808&ccode3=UK&season={season}',
#         'uefa_nation_leagueD': f'https://www.fotmob.com/api/leagues?id=9809&ccode3=UK&season={season}',
#         'uefa_womens_euro': f'https://www.fotmob.com/api/leagues?id=292&ccode3=UK&season={season}',
#         'womens_champions_league': f'https://www.fotmob.com/api/leagues?id=9375&ccode3=UK&season={season}',
#         'womens_world_cuup': f'https://www.fotmob.com/api/leagues?id=76&ccode3=UK&season={season}',
#         'fiffa_world_cup': f'https://www.fotmob.com/api/leagues?id=77&ccode3=UK&season={season}',
#         'world_cup_qualification_concacaf': f'https://www.fotmob.com/api/leagues?id=10198&ccode3=UK&season={season}',
#         'world_cup_qualification_conmebol': f'https://www.fotmob.com/api/leagues?id=10199&ccode3=UK&season={season}',
#         'world_cup_qualification_uefa': f'https://www.fotmob.com/api/leagues?id=10195&ccode3=UK&season={season}',
#     }
#     for league_name, url in leagues_pt.items():
#         if season in leagues[league_name]:
#             leagues_per_season[season] = hml
#             hml.append(league_name)
#     print(f'Season {season} has {len(hml)} leagues: |||||', hml)

In [53]:
for season in seasons:
    leagues_urls = {
        # 'argentinalp': f'https://www.fotmob.com/api/leagues?id=112&ccode3=UK&season={season}',
        # 'copaargentina': f'https://www.fotmob.com/api/leagues?id=9305&ccode3=UK&season={season}',
        # 'copa_de_laliga_argentina': f'https://www.fotmob.com/api/leagues?id=10007&ccode3=UK&season={season}',
        # 'A_league_australia': f'https://www.fotmob.com/api/leagues?id=113&ccode3=UK&season={season}',
        'A_league_australia_women': f'https://www.fotmob.com/api/leagues?id=9495&ccode3=UK&season={season}',
        'bundesliga_austria': f'https://www.fotmob.com/api/leagues?id=38&ccode3=UK&season={season}',
        'first_division_a': f'https://www.fotmob.com/api/leagues?id=40&ccode3=UK&season={season}',
        'serieA_brazil': f'https://www.fotmob.com/api/leagues?id=268&ccode3=UK&season={season}',
        'serieB_brazil': f'https://www.fotmob.com/api/leagues?id=8814&ccode3=UK&season={season}',
        'paulista_A1': f'https://www.fotmob.com/api/leagues?id=10244&ccode3=UK&season={season}',
        'primier_league_canada': f'https://www.fotmob.com/api/leagues?id=9986&ccode3=UK&season={season}',
        'canadian_championship': f'https://www.fotmob.com/api/leagues?id=9837&ccode3=UK&season={season}',
        'primera_division_chile': f'https://www.fotmob.com/api/leagues?id=273&ccode3=UK&season={season}',
        'cup_chile': f'https://www.fotmob.com/api/leagues?id=9091&ccode3=UK&season={season}',
        'super_league_china': f'https://www.fotmob.com/api/leagues?id=120&ccode3=UK&season={season}',
        'primera_A_colombia': f'https://www.fotmob.com/api/leagues?id=274&ccode3=UK&season={season}',
        'HNL_croatia': f'https://www.fotmob.com/api/leagues?id=252&ccode3=UK&season={season}',
        'superligaen_denmark': f'https://www.fotmob.com/api/leagues?id=46&ccode3=UK&season={season}',
        'onedivision_denmark': f'https://www.fotmob.com/api/leagues?id=85&ccode3=UK&season={season}',
        'primier_league_egypt': f'https://www.fotmob.com/api/leagues?id=519&ccode3=UK&season={season}',
        'primier_league_england': f'https://www.fotmob.com/api/leagues?id=47&ccode3=UK&season={season}',
        'championship_england': f'https://www.fotmob.com/api/leagues?id=48&ccode3=UK&season={season}',
        'league_one_england': f'https://www.fotmob.com/api/leagues?id=108&ccode3=UK&season={season}',
        'league_two_england': f'https://www.fotmob.com/api/leagues?id=109&ccode3=UK&season={season}',
        'FA_cup_england': f'https://www.fotmob.com/api/leagues?id=132&ccode3=UK&season={season}',
        'efl_cup_england': f'https://www.fotmob.com/api/leagues?id=133&ccode3=UK&season={season}',
        'wsl_england': f'https://www.fotmob.com/api/leagues?id=9227&ccode3=UK&season={season}',
        'league1_france': f'https://www.fotmob.com/api/leagues?id=53&ccode3=UK&season={season}',
        'league1_france': f'https://www.fotmob.com/api/leagues?id=110&ccode3=UK&season={season}',
        'copa_de_france': f'https://www.fotmob.com/api/leagues?id=134&ccode3=UK&season={season}',
        'bundesliga_germany': f'https://www.fotmob.com/api/leagues?id=54&ccode3=UK&season={season}',
        'bundesliga2_germany': f'https://www.fotmob.com/api/leagues?id=146&ccode3=UK&season={season}',
        'liga3_germany': f'https://www.fotmob.com/api/leagues?id=208&ccode3=UK&season={season}',
        'dfb_pokal_germany': f'https://www.fotmob.com/api/leagues?id=209&ccode3=UK&season={season}',
        'Frauen_Bundesliga': f'https://www.fotmob.com/api/leagues?id=9676&ccode3=UK&season={season}',
        'super_league1_greece': f'https://www.fotmob.com/api/leagues?id=135&ccode3=UK&season={season}',
        'besta_deildin_iceland': f'https://www.fotmob.com/api/leagues?id=215&ccode3=UK&season={season}',
        'super_league_india': f'https://www.fotmob.com/api/leagues?id=9478&ccode3=UK&season={season}',
        'premier_division_ireland':  f'https://www.fotmob.com/api/leagues?id=126&ccode3=UK&season={season}',
        'serieA_italy': f'https://www.fotmob.com/api/leagues?id=55&ccode3=UK&season={season}',
        'serieB_italy': f'https://www.fotmob.com/api/leagues?id=86&ccode3=UK&season={season}',
        'coppa_italia_italy': f'https://www.fotmob.com/api/leagues?id=141&ccode3=UK&season={season}',
        'jleague_japan': f'https://www.fotmob.com/api/leagues?id=223&ccode3=UK&season={season}',
        'ligamx_mexico': f'https://www.fotmob.com/api/leagues?id=230&ccode3=UK&season={season}',
        'Eredivisie_netherland': f'https://www.fotmob.com/api/leagues?id=57&ccode3=UK&season={season}',
        'eerste_divisie_netherland': f'https://www.fotmob.com/api/leagues?id=111&ccode3=UK&season={season}',
        'knvb_cup_netherland': f'https://www.fotmob.com/api/leagues?id=235&ccode3=UK&season={season}',
        'elisteserien': f'https://www.fotmob.com/api/leagues?id=59&ccode3=UK&season={season}',
        'Ekstraklasa_poland': f'https://www.fotmob.com/api/leagues?id=196&ccode3=UK&season={season}',
        'liga_portugal': f'https://www.fotmob.com/api/leagues?id=61&ccode3=UK&season={season}',
        'premier_league_russia': f'https://www.fotmob.com/api/leagues?id=63&ccode3=UK&season={season}',
        'kings_cup_KSA': f'https://www.fotmob.com/api/leagues?id=9942&ccode3=UK&season={season}',
        'premiership_scotland': f'https://www.fotmob.com/api/leagues?id=64&ccode3=UK&season={season}',
        'k_league1_skorea': f'https://www.fotmob.com/api/leagues?id=9080&ccode3=UK&season={season}',
        'k_league2_skorea': f'https://www.fotmob.com/api/leagues?id=9116&ccode3=UK&season={season}',
        'laliga_spain': f'https://www.fotmob.com/api/leagues?id=87&ccode3=UK&season={season}',
        'laliga2_spain': f'https://www.fotmob.com/api/leagues?id=140&ccode3=UK&season={season}',
        'laligaf_spain': f'https://www.fotmob.com/api/leagues?id=9907&ccode3=UK&season={season}',
        'copa_del_rey_spain': f'https://www.fotmob.com/api/leagues?id=138&ccode3=UK&season={season}',
        'Allsvenskan_sweeden': f'https://www.fotmob.com/api/leagues?id=67&ccode3=UK&season={season}',
        'super_league_switzerland': f'https://www.fotmob.com/api/leagues?id=69&ccode3=UK&season={season}',
        'thai_league_thailand': f'https://www.fotmob.com/api/leagues?id=8984&ccode3=UK&season={season}',
        'super_league_turkey': f'https://www.fotmob.com/api/leagues?id=71&ccode3=UK&season={season}',
        'proleagueUAE': f'https://www.fotmob.com/api/leagues?id=538&ccode3=UK&season={season}',
        'MLS_USA': f'https://www.fotmob.com/api/leagues?id=130&ccode3=UK&season={season}',
        'usl_championship_usa': f'https://www.fotmob.com/api/leagues?id=8972&ccode3=UK&season={season}',
        'usl_league_one': f'https://www.fotmob.com/api/leagues?id=9296&ccode3=UK&season={season}',
        'open_cup_usa': f'https://www.fotmob.com/api/leagues?id=9441&ccode3=UK&season={season}',
        'nwsl_usa': f'https://www.fotmob.com/api/leagues?id=9134&ccode3=UK&season={season}',
        'mls_nextpro_usa': f'https://www.fotmob.com/api/leagues?id=10282&ccode3=UK&season={season}',
        'champions_league': f'https://www.fotmob.com/api/leagues?id=42&ccode3=UK&season={season}',
        'europa_league': f'https://www.fotmob.com/api/leagues?id=73&ccode3=UK&season={season}',
        'afc_champions_league': f'https://www.fotmob.com/api/leagues?id=525&ccode3=UK&season={season}',
        'asian_cup': f'https://www.fotmob.com/api/leagues?id=290&ccode3=UK&season={season}',
        'caf_champions_league': f'https://www.fotmob.com/api/leagues?id=526&ccode3=UK&season={season}',
        'caf_confed_cup': f'https://www.fotmob.com/api/leagues?id=9468&ccode3=UK&season={season}',
        'cocacaf_champ': f'https://www.fotmob.com/api/leagues?id=297&ccode3=UK&season={season}',
        'concacaf_gold': f'https://www.fotmob.com/api/leagues?id=298&ccode3=UK&season={season}',
        'conmebol': f'https://www.fotmob.com/api/leagues?id=10304&ccode3=UK&season={season}',
        'copa_america': f'https://www.fotmob.com/api/leagues?id=44&ccode3=UK&season={season}',
        'copa_libertadores': f'https://www.fotmob.com/api/leagues?id=45&ccode3=UK&season={season}',
        'copa_sudamericana': f'https://www.fotmob.com/api/leagues?id=299&ccode3=UK&season={season}',
        'euro': f'https://www.fotmob.com/api/leagues?id=50&ccode3=UK&season={season}',
        'eurou21': f'https://www.fotmob.com/api/leagues?id=288&ccode3=UK&season={season}',
        'europa_conference': f'https://www.fotmob.com/api/leagues?id=10216&ccode3=UK&season={season}',
        'fifa_clup_world_cup': f'https://www.fotmob.com/api/leagues?id=78&ccode3=UK&season={season}',
        'leagues_cup_int': f'https://www.fotmob.com/api/leagues?id=10043&ccode3=UK&season={season}',
        'uefa_nation_leagueA': f'https://www.fotmob.com/api/leagues?id=9806&ccode3=UK&season={season}',
        'uefa_nation_leagueB': f'https://www.fotmob.com/api/leagues?id=9807&ccode3=UK&season={season}',
        'uefa_nation_leagueC': f'https://www.fotmob.com/api/leagues?id=9808&ccode3=UK&season={season}',
        'uefa_nation_leagueD': f'https://www.fotmob.com/api/leagues?id=9809&ccode3=UK&season={season}',
        'uefa_womens_euro': f'https://www.fotmob.com/api/leagues?id=292&ccode3=UK&season={season}',
        'womens_champions_league': f'https://www.fotmob.com/api/leagues?id=9375&ccode3=UK&season={season}',
        'womens_world_cuup': f'https://www.fotmob.com/api/leagues?id=76&ccode3=UK&season={season}',
        'fiffa_world_cup': f'https://www.fotmob.com/api/leagues?id=77&ccode3=UK&season={season}',
        'world_cup_qualification_concacaf': f'https://www.fotmob.com/api/leagues?id=10198&ccode3=UK&season={season}',
        'world_cup_qualification_conmebol': f'https://www.fotmob.com/api/leagues?id=10199&ccode3=UK&season={season}',
        'world_cup_qualification_uefa': f'https://www.fotmob.com/api/leagues?id=10195&ccode3=UK&season={season}',
    }
    for league_name, url in leagues_urls.items():
        if season in leagues[league_name]:
            print(url)
            # print(leagues[league_name])
            print(f'********** This is the league {league_name} and season {season}. **********')
            ids = all_matches_data(url)
            single_league = match_details(ids)
            df = pd.DataFrame(single_league)
            df.to_csv(f"Fotmob/{league_name}_{season}.csv", mode='w', index=False, encoding="utf-8")


https://www.fotmob.com/api/leagues?id=9495&ccode3=UK&season=2022%2F2023
********** This is the league A_league_australia_women and season 2022%2F2023. **********
https://www.fotmob.com/api/matchDetails?matchId=4026774
The remain ids are  101
------------------------------------------------------------
https://www.fotmob.com/api/matchDetails?matchId=4026758
The remain ids are  100
------------------------------------------------------------
https://www.fotmob.com/api/matchDetails?matchId=4026773
The remain ids are  99
------------------------------------------------------------
https://www.fotmob.com/api/matchDetails?matchId=4026821
The remain ids are  98
------------------------------------------------------------
https://www.fotmob.com/api/matchDetails?matchId=4026772
The remain ids are  97
------------------------------------------------------------
https://www.fotmob.com/api/matchDetails?matchId=4026820
The remain ids are  96
---------------------------------------------------------

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))